# SCENARIO 3: STREAMING (JSON)

An e-commerce platform receives customer order details from its mobile application in JSON format through a streaming pipeline. The JSON contains nested fields such as customer information, payment details, and a list of purchased items. To store and analyze this data efficiently in a data warehouse, the nested structure must be flattened into a tabular format using PySpark, ensuring all relevant attributes are readily accessible for reporting and analytics.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = (spark.read
      .format('json')
      .option('inferSchema', True)
      .option('multiLine', True)
      .load('/Volumes/pyspark_catalog/source/db_volume/jsonData/'))

display(df)
df.printSchema()

customer,delivery_updates,items,order_id,order_timestamp,payment
"List(CUST101, john.doe@example.com, List(Toronto, Canada), John Doe)","List(Order Placed, Packed, Shipped, Out for Delivery)","List(List(ITEM1001, 25.5, Wireless Mouse, 2), List(ITEM1002, 199.75, Mechanical Keyboard, 1))",ORD001,2025-08-15T10:45:30Z,"List(250.75, CAD, Credit Card)"
"List(CUST102, jane.smith@example.com, List(Vancouver, Canada), Jane Smith)","List(Order Placed, Packed, Shipped)","List(List(ITEM1003, 89.99, USB-C Hub, 1))",ORD002,2025-08-15T11:10:15Z,"List(89.99, CAD, PayPal)"


root
 |-- customer: struct (nullable = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- location: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- delivery_updates: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- price_per_unit: double (nullable = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- method: string (nullable = true)



## Querying Data

In [0]:
# We explode when column's type is array

## One order with multiple items becomes multiple rows
df_flat = df.withColumn("item", explode("items"))

## Select and flatten the nested fields
df_flat = df_flat.select(
    
    # Order information
    col("order_id"),
    col("order_timestamp"),

    # Customer information
    col("customer.customer_id"),
    col("customer.email").alias("customer_email"),

    # Customer location infor
    col("customer.location.city").alias("customer_city"),
    col("customer.location.country").alias("customer_country"),

    # Payment information
    col("payment.amount").alias("payment_amount"),
    col("payment.currency").alias("payment_currency"),
    col("payment.method").alias("payment_method"),

    # Item information (from the column item exploded above)
    col("item.item_id").alias("item_id"),
    col("item.product_name").alias("product_name"),
    col("item.price_per_unit").alias("price_per_unit"),
    col("item.quantity").alias("quantity")
)

display(df_flat)

# If we need delivery history for analytics, the better design is usually a separate delivery-history table.


order_id,order_timestamp,customer_id,customer_email,customer_city,customer_country,payment_amount,payment_currency,payment_method,item_id,product_name,price_per_unit,quantity
ORD001,2025-08-15T10:45:30Z,CUST101,john.doe@example.com,Toronto,Canada,250.75,CAD,Credit Card,ITEM1001,Wireless Mouse,25.5,2
ORD001,2025-08-15T10:45:30Z,CUST101,john.doe@example.com,Toronto,Canada,250.75,CAD,Credit Card,ITEM1002,Mechanical Keyboard,199.75,1
ORD002,2025-08-15T11:10:15Z,CUST102,jane.smith@example.com,Vancouver,Canada,89.99,CAD,PayPal,ITEM1003,USB-C Hub,89.99,1
